In [1]:
!pip install torchmetrics
!pip install pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.8/427.8 kB 18.6 MB/s eta 0:00:00


In [2]:
import torch
import torchmetrics

import torchvision

from torchvision.models.detection.faster_rcnn import fasterrcnn_resnet50_fpn

import cv2

import numpy as np

import os

import sklearn

import random

import json

from PIL import Image

from torch.utils.data import DataLoader, Dataset

In [5]:
from torchvision.models.detection.faster_rcnn import fasterrcnn_resnet50_fpn, FastRCNNPredictor


def get_model(num_classes):

    model = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=800, max_size=800)

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

In [6]:
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision
def testModel(model,dataloader,pred=[],rcl=[],acry=[],mAP_list=[],mAP75_list=[]):
    model.to(device)
    model.eval()
    # box_lists = []
    # label_lists = []
    score_lists = []
    # targets =[]
    metric = MeanAveragePrecision()
    total_TP = 0
    total_FP = 0
    total_FN = 0
    for ims,tgs in dataloader:
                
                targets_box =[]
                targets_label = []
        
                images = list(image.to(device) for image in ims)
                
                targets = [{k: v.to(device) for k, v in t.items()} for t in tgs]
                for target in targets:
                        targets_box.append(target['boxes'])
                        targets_label.append(target['labels'])
                i=0
                with torch.no_grad():
                    predictions = model(images)
                    results = []
                    targets_list = []
                    for output in (predictions):
                        boxes = output['boxes']
                        scores = output['scores']
                        labels = output['labels']
            
                        # Áp dụng NMS xóa các bouding box chồng lên nhau
                        nms_threshold = 0.5
                        keep_indices = torchvision.ops.nms(boxes, scores, nms_threshold)
    
                        # print(keep_indices)
                        # 
                        filtered_boxes = boxes[keep_indices]
                        filtered_scores = scores[keep_indices]
                        filtered_labels = labels[keep_indices]
                        # Dự đoán (boxes, labels, scores)
                        
                        tp,fp,fn = Caculate_TP(filtered_boxes,targets_box[i])   
                        total_TP += tp
                        total_FP += fp
                        total_FN += fn
                        # Thêm vào các predictions và targets để tính mAP
                        results.append({
                            'boxes': filtered_boxes,
                            'scores': filtered_scores,
                            'labels': filtered_labels
                        })
                        targets_list.append({
                            'boxes': targets_box[i],
                            'labels': targets_label[i]
                        })
                        i +=1 
                    metric.update(preds=results, target=targets_list)
    # Tính mAP sau khi kết thúc
    mAP = metric.compute()
    print(f"mAP: {mAP}")
    print(f"TP: {total_TP},FP: {total_FP},FN: {total_FN}")
    
    # Tính Precision, Recall, Accuracy
    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
    recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
    accuracy = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0

    pred.append(precision)
    rcl.append(recall) 
    acry.append(accuracy)
    mAP_list.append(mAP['map_50'])
    mAP75_list.append(mAP['map_75'])
    
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, Accuracy: {accuracy:.4f}")


In [7]:
from torchvision.ops import box_iou

def Caculate_TP(pred_box,target_box):
    iou_threshold = 0.5
    
    # Khởi tạo TP, FP, FN
    TP = 0
    FP = 0
    FN = 0
    
    # tính iou_scores
    iou_scores = box_iou(pred_box, target_box)

    # Duyệt qua từng box dự đoán và kiểm tra nếu có IoU >= ngưỡng với một Ground Truth box
    for i in range(iou_scores.size(0)):
        if (iou_scores[i] >= iou_threshold).any():  # Nếu box dự đoán i có IoU >= ngưỡng với bất kỳ Ground Truth box nào
            TP += 1
        else:
            FP += 1
    
    # Kiểm tra FN cho từng Ground Truth box
    for j in range(iou_scores.size(1)):
        if (iou_scores[:, j] < iou_threshold).all():  # Nếu không có box dự đoán nào đạt ngưỡng với Ground Truth box j
            FN += 1
    # print(TP,FP,FN)
    return TP,FP,FN

DỰ ĐOÁN HÌNH ẢNH

In [20]:
import torch
import torchvision
from torchvision import transforms as T

from PIL import Image
import cv2
import matplotlib.pyplot as plt

className = ['pl80', 'pl5', 'pl120', 'p26', 'il60', 'pl60', 'pl100', 'pl30', 'p12',
                'il50', 'il90', 'ph5', 'pn', 'w57', 'p27', 'p18', 'i4', 'pm20', 'p13',
                'il110', 'il80', 'pne', 'pm30', 'pss', 'pa13', 'i2', 'p9', 'p10', 'p29',
                'i5', 'p11', 'pmb', 'pr40', 'p6', 'pl20', 'pl110', 'p17', 'p5', 'ip', 'w3',
                'pl40', 'pr20', 'pr50', 'p23', 'p3', 'ph4.2', 'pcr', 'p16', 'iz', 'p8',
                'ph2.5', 'pb', 'ph3', 'ph2.2', 'il70', 'pr30', 'pl50', 'i2r', 'p1n', 'wc',
                'pbp', 'ph4.5', 'i4l', 'p14', 'pm10', 'pw4', 'i13', 'pw3.2', 'pmr', 'ph3.5',
                'pbm', 'pm55', 'il100', 'p2', 'pm15', 'w20', 'pctl', 'pl70', 'pl10', 'w30',
                'pl90', 'pr60', 'ph4', 'pcd', 'w45', 'pg', 'w22', 'w55', 'pl15', 'w13', 'ph2',
                'w42', 'pm35', 'p15', 'w34', 'pl35', 'p4', 'pdd', 'w32', 'pm2', 'pr70', 'pr80',
                'pmblr', 'w35', 'w59', 'pcl', 'p19', 'pa12', 'w41', 'w18', 'phcs', 'i14',
                'pm40', 'w38', 'w12', 'pw3', 'pnlc', 'pa10', 'p25', 'w63', 'p1', 'i10',
                'w47', 'phclr', 'pt', 'pl25', 'pm50', 'w37', 'ph3.2', 'ph2.4', 'im', 'w58',
                'pa14', 'ps', 'w21', 'i12', 'i1', 'w46', 'pn-2', 'pm5', 'ph4.8', 'ph4.3',
                'i3', 'ph2.8', 'pw3.5', 'pm8', 'p28', 'ph1.8', 'ph2.1']
def predic(model, input_path):
    im = Image.open(input_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device) 
    transform = T.Compose([
        T.Resize((800, 800)),  # Resize 800x800
        T.ToTensor()           
    ])
    img = transform(im)
    img = img.to(device)  
    model.eval()
    with torch.no_grad():
        pred = model([img])
    
    boxes, labels, scores = pred[0]["boxes"], pred[0]["labels"], pred[0]["scores"]
    
    keep_indices = torchvision.ops.nms(boxes, scores, 0.4)
    boxes = boxes[keep_indices]
    scores = scores[keep_indices]
    labels = labels[keep_indices]
    
    num = torch.argwhere(scores > 0.5).shape[0]
    
    igg = cv2.imread(input_path)
    font = cv2.FONT_HERSHEY_SIMPLEX
    igg = cv2.cvtColor(igg, cv2.COLOR_BGR2RGB)  # Chuyển BGR sang RGB
    
    # Resize  800x800 to 2048x2048
    old_width, old_height = 800, 800
    new_width, new_height = 2048, 2048
    scale_x = new_width / old_width
    scale_y = new_height / old_height
    
    igg_resized = cv2.resize(igg, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    
    for i in range(num):
        x1, y1, x2, y2 = boxes[i].cpu().numpy().astype('int')
        
        # Scale tọa độ của box theo tỉ lệ resize
        x1 = int(x1 * scale_x)
        y1 = int(y1 * scale_y)
        x2 = int(x2 * scale_x)
        y2 = int(y2 * scale_y)
        
        label_name = className[labels.cpu().numpy()[i] - 1]
        igg_resized = cv2.rectangle(igg_resized, (x1, y1), (x2, y2), (0, 255, 0), 2)
        igg_resized = cv2.putText(igg_resized, label_name, (x1, y1 - 5), font, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
    
    
    filename = os.path.basename(input_path)
    path_save = "/kaggle/working/predic"
    output = os.path.join(path_save, filename)
    os.makedirs(os.path.dirname(output), exist_ok=True)

    
    cv2.imwrite(output, cv2.cvtColor(igg_resized, cv2.COLOR_RGB2BGR))  # convert RGB to BGR before save

In [21]:
model = get_model(150)
model.load_state_dict(torch.load("/kaggle/input/frcnn-1e-3/best_model.pth"))
count = 0
source = []
folder_path = "/kaggle/input/tt100kv1/tt100kv1/images/test"


for filename in os.listdir(folder_path):
    count +=1
    if count >50:
        break
    if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
        input_path = os.path.join(folder_path, filename)
        predic(model,input_path)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 205MB/s]
/tmp/ipykernel_23/1456124643.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you